# qldd on Colab

Small-scale runs of the local diffusion decoder: stage-1 go/no-go at d=3, plus
the d=7 windowed locality run that doesn't fit on a CPU.

Set the runtime to GPU (T4 is enough), drag `qldd.zip` into the Files pane,
then run all cells.


In [ ]:
# 1. deps (Colab already has a GPU torch; we add the QEC libs)
!pip -q install stim pymatching
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


In [ ]:
# 2. get the qldd source: unzip the uploaded qldd.zip (or git clone your repo)
import os, sys, glob
if not os.path.isdir('qldd/src/qldd'):
    zips = sorted(glob.glob('qldd*.zip'))
    if zips:
        !unzip -oq {zips[-1]} -d .
    # else: !git clone https://github.com/<you>/qldd.git
assert os.path.isdir('qldd/src/qldd'), 'upload qldd.zip via the Files pane first'
sys.path.insert(0, 'qldd/src')
import qldd; print('qldd ready')


## Sanity: data contract + MWPM threshold
Confirms `s=He`, `l=Le` and the MWPM crossing near p*~0.04. CPU, seconds.


In [ ]:
from qldd.data import make_code_data, verify_contract
from qldd.baseline import threshold_sweep
for d in (3,5,7):
    r = verify_contract(make_code_data(distance=d, p=0.02))
    print(f'd={d}: s=He {r["s_equals_He"]}  l=Le {r["l_equals_Le"]}  n_err={r["n_err"]}')
sw = threshold_sweep([3,5,7], [0.01,0.03,0.05,0.08], shots=20000)
print('p:', sw['ps'])
for d in sw['distances']: print(f'd={d}:', [round(x,4) for x in sw['ler'][d]])


## Stage 1: does diffusion-over-`e` reach MWPM at d=3 (global limit)?
Converges in a couple of minutes on a T4.


In [ ]:
import numpy as np, torch
from qldd.data import make_code_data, sample
from qldd.model import LocalDiffusionDecoder, ModelConfig
from qldd.diffusion import diffusion_loss, evaluate_ler
from qldd.baseline import build_matching, logical_error_rate
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
code = make_code_data(distance=3, p=0.03)
mwpm = logical_error_rate(code, 50000, build_matching(code), seed=7)['ler']
m = LocalDiffusionDecoder(ModelConfig(d_model=128,n_heads=8,n_layers=4,d_ff=512,
        use_conv_stem=False, sigma_init=100.0), code).to(dev)  # global limit
opt = torch.optim.AdamW(m.parameters(), lr=3e-3, weight_decay=1e-4)
hist = []
for step in range(1, 6001):
    e,s,l = sample(code, 512)
    st = torch.as_tensor(s, dtype=torch.long, device=dev)
    ee = torch.as_tensor(e, dtype=torch.long, device=dev)
    loss = diffusion_loss(m, st, ee); opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        e,s,l = sample(code, 20000, seed=9)
        r = evaluate_ler(m, code, s, e, l, n_steps=16, device=dev, batch=2048)
        hist.append((step, r['ler'], r['ler_strict'], r['syndrome_cleared_frac']))
        print(f'step {step}: LER {r["ler"]:.4f}  strict {r["ler_strict"]:.4f}  '
              f'cleared {r["syndrome_cleared_frac"]:.3f}  | MWPM {mwpm:.4f}')


In [ ]:
import matplotlib.pyplot as plt
h = np.array(hist)
plt.figure(); plt.plot(h[:,0],h[:,1],label='diffusion LER')
plt.plot(h[:,0],h[:,2],label='strict LER (non-clear=fail)')
plt.axhline(mwpm, ls='--', c='k', label='MWPM'); plt.xlabel('step'); plt.ylabel('LER')
plt.yscale('log'); plt.legend(); plt.title('Stage 1: reach MWPM?'); plt.show()
plt.figure(); plt.plot(h[:,0],h[:,3]); plt.xlabel('step')
plt.ylabel('syndrome cleared fraction'); plt.ylim(0,1); plt.show()
go = (h[-1,1] <= 1.1*mwpm) and (h[-1,3] > 0.99)
print('VERDICT:', 'GO -> proceed to locality study' if go else 'NO-GO -> rethink architecture')


## d=7 windowed, conv off, trainable locality
Sparse local attention (`window_radius`) makes d=7 fit a T4. Watch total range
vs grid width: if it stays small as d grows, the local recovery map holds.


In [ ]:
# d=7 windowed, conv off, trainable locality. T4-safe: small radius keeps K (and
# the (B,h,T,K,dk) gather) modest. Raise window_radius / d_model on an A100.
from qldd.analysis import locality_report
code7 = make_code_data(distance=7, p=0.03)
mwpm7 = logical_error_rate(code7, 50000, build_matching(code7), seed=7)['ler']
m7 = LocalDiffusionDecoder(ModelConfig(d_model=128, n_heads=4, n_layers=4, d_ff=384,
        use_conv_stem=False, sigma_init=1.0, window_radius=1.5), code7).to(dev)
print('window K =', m7.window_K, 'of T =', code7.n_det+code7.n_err,
      '| params', sum(p.numel() for p in m7.parameters()))
opt = torch.optim.AdamW(m7.parameters(), lr=2e-3, weight_decay=1e-4)
BATCH7 = 32   # raise on bigger GPUs; gather mem ~ BATCH*K*dk
for step in range(1, 8001):
    e,s,l = sample(code7, BATCH7)
    st = torch.as_tensor(s, dtype=torch.long, device=dev)
    ee = torch.as_tensor(e, dtype=torch.long, device=dev)
    loss = diffusion_loss(m7, st, ee); opt.zero_grad(); loss.backward(); opt.step()
    if step % 1000 == 0:
        e,s,l = sample(code7, 20000, seed=9)
        r = evaluate_ler(m7, code7, s, e, l, n_steps=24, device=dev, batch=128)
        loc = locality_report(m7, code7)
        print(f'step {step}: LER {r["ler"]:.4f}  cleared {r["syndrome_cleared_frac"]:.3f}  '
              f'range {loc["total_effective_range_lattice"]:.1f}/{loc["grid_width_voxels"]}  '
              f'meaningful {loc["locality_test_meaningful"]}  | MWPM {mwpm7:.4f}')


Next: sweep `sigma_init`, shrink `window_radius` to find where accuracy breaks,
vary inference `n_steps`, push p toward threshold, try d=9. `make_ablation.py`
generates the full conv_layers x sigma_init grid.
